# Comparación de Resolución Temporal y Degradación de Datos: MATLAB vs. Veraset

Este notebook tiene como objetivo cuantificar las diferencias estadísticas de resolución temporal ($\Delta t$) entre el dataset controlado de **MATLAB** (alta frecuencia, ~1s) y el de **Veraset** (baja frecuencia, datos de la vida real).

Posteriormente, se presenta un prototipo de función de **degradación de trayectorias** para simular la resolución de Veraset en los datos de MATLAB. Esto permitirá entrenar el clasificador bayesiano en un escenario de baja resolución realista.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Definición de Rutas y Carga de Datos

In [ ]:
# Rutas relativas del proyecto
PROJECT_ROOT = Path("../../")
GPS_DATA_DIR = PROJECT_ROOT / "Inputs" / "GPS User Data"

PATH_MATLAB = GPS_DATA_DIR / "Datos de MATLAB GPS.csv"
PATH_VERASET = GPS_DATA_DIR / "top_20users_1_month.parquet"

print("Cargando muestra de MATLAB...")
df_matlab = pd.read_csv(PATH_MATLAB, nrows=50000)  # Cargar primeras filas para análisis rápido

print("Cargando muestra de Veraset...")
df_veraset = pd.read_parquet(PATH_VERASET)

print(f"MATLAB cargado con {len(df_matlab)} filas.")
print(f"Veraset cargado con {len(df_veraset)} filas.")

## 2. Análisis del Intervalo Temporal ($\Delta t$)

In [ ]:
# Convertir a datetime
df_matlab['datetime'] = pd.to_datetime(df_matlab['Timestamp'])
df_veraset['datetime'] = pd.to_datetime(df_veraset['utc_timestamp'], unit='s')

# Ordenar cronológicamente por usuario/viaje
df_matlab = df_matlab.sort_values(by=['caid', 'num_trip', 'datetime'])
df_veraset = df_veraset.sort_values(by=['caid', 'datetime'])

# Calcular deltas de tiempo en segundos
df_matlab['delta_t'] = df_matlab.groupby(['caid', 'num_trip'])['datetime'].diff().dt.total_seconds()
df_veraset['delta_t'] = df_veraset.groupby('caid')['datetime'].diff().dt.total_seconds()

# Filtrar deltas de 0 y deltas gigantescos que indiquen corte entre días/viajes
df_matlab_clean = df_matlab[df_matlab['delta_t'] > 0]
df_veraset_clean = df_veraset[(df_veraset['delta_t'] > 0) & (df_veraset['delta_t'] < 3600)]  # max 1 hora

print("--- Estadísticas de Delta t (Segundos) en MATLAB ---")
print(df_matlab_clean['delta_t'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95]))

print("\n--- Estadísticas de Delta t (Segundos) en VERASET ---")
print(df_veraset_clean['delta_t'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95]))

### Gráfica de Distribución de $\Delta t$

In [ ]:
plt.figure(figsize=(12, 6))
sns.kdeplot(data=df_matlab_clean, x='delta_t', label='MATLAB', fill=True, alpha=0.5, bw_adjust=1.5)
sns.kdeplot(data=df_veraset_clean, x='delta_t', label='Veraset', fill=True, alpha=0.5, bw_adjust=1.5)
plt.title('Comparación de Distribución de Intervalos Temporales ($\Delta t$)')
plt.xlabel('Intervalo de Tiempo (Segundos)')
plt.ylabel('Densidad')
plt.xlim(0, 300)  # Limitar para mejor visualización
plt.legend()
plt.show()

## 3. Prototipo de Degradador de Datos (Simulador de Veraset)

Para transformar una trayectoria continua de MATLAB en una trayectoria ruidosa y dispersa tipo Veraset:
1. **Muestreo por Intervalo:** En lugar de tomar pings continuos, se simula una tasa promedio mayor.
2. **Generación de Baches (Signal Loss Gaps):** Se introduce una probabilidad de desconexión y una duración aleatoria de pérdida de señal.

In [ ]:
def degradar_trayectoria(df_trip, delta_promedio_sec=30, prob_gap=0.05, max_gap_size=10):
    """
    Degrada una trayectoria de MATLAB para asemejarla a Veraset.
    
    Parameters:
    - df_trip: DataFrame de un viaje específico de MATLAB ordenado cronológicamente.
    - delta_promedio_sec: Tiempo promedio objetivo entre pings (en segundos).
    - prob_gap: Probabilidad de que ocurra una interrupción de señal en cada punto.
    - max_gap_size: Número máximo de pings consecutivos a eliminar en un bache.
    
    Returns:
    - DataFrame del viaje degradado.
    """
    if len(df_trip) < 5:
        return df_trip
    
    # 1. Submuestreo temporal aproximado (ej: si MATLAB es 1s, y queremos 30s, tomamos 1 de cada 30)
    indices_a_conservar = [0]  # Siempre conservar el inicio
    ultimo_tiempo = df_trip['datetime'].iloc[0]
    
    for idx in range(1, len(df_trip) - 1):
        tiempo_actual = df_trip['datetime'].iloc[idx]
        elapsed = (tiempo_actual - ultimo_tiempo).total_seconds()
        
        # Si ha pasado suficiente tiempo, evaluamos conservarlo
        if elapsed >= delta_promedio_sec:
            indices_a_conservar.append(idx)
            ultimo_tiempo = tiempo_actual
            
    indices_a_conservar.append(len(df_trip) - 1)  # Siempre conservar el final
    df_degraded = df_trip.iloc[indices_a_conservar].copy()
    
    # 2. Inyección de baches (Gaps)
    df_degraded = df_degraded.reset_index(drop=True)
    rows_to_drop = set()
    i = 0
    while i < len(df_degraded) - 2:
        if i > 0 and np.random.rand() < prob_gap:
            gap_size = np.random.randint(1, max_gap_size + 1)
            for drop_idx in range(i, min(i + gap_size, len(df_degraded) - 1)):
                rows_to_drop.add(drop_idx)
            i += gap_size
        else:
            i += 1
            
    df_degraded = df_degraded.drop(list(rows_to_drop)).reset_index(drop=True)
    return df_degraded

## 4. Visualización del Impacto de la Degradación

Seleccionamos un viaje de MATLAB específico para visualizar los puntos originales frente a los degradados.

In [ ]:
# Seleccionar un viaje representativo (por ejemplo, con suficientes puntos)
trips = df_matlab['num_trip'].dropna().unique()
if len(trips) > 0:
    sample_trip_id = trips[0]
    df_sample_trip = df_matlab[df_matlab['num_trip'] == sample_trip_id].copy()

    # Aplicar la degradación
    np.random.seed(42)
    df_sample_degraded = degradar_trayectoria(df_sample_trip, delta_promedio_sec=30, prob_gap=0.08, max_gap_size=5)

    print(f"Viaje de muestra {sample_trip_id}:")
    print(f"- Puntos originales (MATLAB): {len(df_sample_trip)}")
    print(f"- Puntos degradados (Simulado Veraset): {len(df_sample_degraded)}")

    # Graficar los puntos en el espacio
    plt.figure(figsize=(10, 8))
    plt.plot(df_sample_trip['lon'], df_sample_trip['lat'], 'b-', label='Trayectoria Original (MATLAB)', alpha=0.3)
    plt.scatter(df_sample_trip['lon'], df_sample_trip['lat'], c='blue', s=10, label='Puntos MATLAB', alpha=0.5)
    plt.scatter(df_sample_degraded['lon'], df_sample_degraded['lat'], c='red', s=40, label='Puntos Degradados (Veraset Sintético)', marker='x')
    plt.title(f'Degradación Espacial del Viaje {sample_trip_id}')
    plt.xlabel('Longitud')
    plt.ylabel('Latitud')
    plt.legend()
    plt.show()
else:
    print("No se encontraron viajes válidos en la muestra cargada.")